In [15]:
# get table
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
utils.fetch_data_from_postgres_via_psycopg2(
  """
  SELECT table_name
  FROM information_schema.tables
  WHERE table_schema = 'public';
"""
)
# user_id: [1, 6040]
# movie_id: [1, 3706]

,table_name
0,tpcxai_order_training
1,tpcxai_product_serving
2,tpcxai_financial_transactions_serving
3,tpcxai_store_dept_serving
4,tpcxai_order_serving
5,tpcxai_financial_account_training
6,tpcxai_lineitem_training
7,tpcxai_financial_account_serving
8,tpcxai_product_rating_training
9,tpcxai_lineitem_serving


In [ ]:
# template 4
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
from sklearn.model_selection import train_test_split

# load data
df_user = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_user""")
df_movie = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_movie""")
df_rating = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_rating""")

from sklearn.preprocessing import LabelEncoder, MinMaxScaler

user_min_max_scaler = MinMaxScaler()
gender_encoder = LabelEncoder()
df_user['u_gender_encoded']  = gender_encoder.fit_transform(df_user['u_gender'])
df_user[['u_age_encoded', 'u_occupation_encoded']] = user_min_max_scaler.fit_transform(df_user[['u_age','u_occupation']])
# prepare one-hot encoding for movie genres
list_genres = df_movie['m_genres'].str.split('|').tolist()
unique_genres = set(genre for sublist in list_genres for genre in sublist)
df_movie['m_genres_encoded'] = df_movie['m_genres'].apply(
    lambda x: [1 if genre in x else 0 for genre in unique_genres]
)

# training part
df_train = df_rating.merge(df_user, left_on='r_user_id', right_on='u_user_id')
df_train = df_train.merge(df_movie, left_on='r_movie_id', right_on='m_movie_id')
rating_min_max_scaler = MinMaxScaler()
df_train['r_rating_encoded'] = rating_min_max_scaler.fit_transform(df_train[['r_rating']])

X = np.hstack([
  df_train[["u_gender_encoded", "u_age_encoded", "u_occupation_encoded"]].values,
  np.vstack(df_train["m_genres_encoded"].values)
])
y = df_train["r_rating_encoded"].values

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
# model.fit(X_train, y_train, epochs=10, batch_size=2560, validation_data=(X_val, y_val))

# testing part
# df_test = df_user.iloc[:10].merge(df_movie, how='cross')
# # df_test = df_user.merge(df_movie, how='cross')
# X_infer = np.hstack([
#   df_test[["u_gender_encoded", "u_age_encoded", "u_occupation_encoded"]].values,
#   np.vstack(df_test["m_genres_encoded"].values)
# ])
# y_pred = model.predict(X_infer)

2025-06-19 22:06:49.462051: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-19 22:06:49.503874: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-19 22:06:49.503929: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-19 22:06:49.505265: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-19 22:06:49.512438: I tensorflow/core/platform/cpu_feature_guar

1159/1159 [==============================] - 1s 957us/step


In [8]:
df_test = utils.fetch_data_from_postgres_via_psycopg2(
  """
  select u_user_id, m_movie_id, u_gender, u_age, u_occupation, m_genres
  from movielens_user
  cross join 
  movielens_movie
  limit 10;
""")
df_test['u_gender_encoded'] = gender_encoder.transform(df_test['u_gender'])
df_test[['u_age_encoded', 'u_occupation_encoded']] = user_min_max_scaler.transform(df_test[['u_age', 'u_occupation']])
df_test['m_genres_encoded'] = df_test['m_genres'].apply(
    lambda x: [1 if genre in x else 0 for genre in unique_genres]
)
X_infer = np.hstack([
  df_test[["u_gender_encoded", "u_age_encoded", "u_occupation_encoded"]].values,
  np.vstack(df_test["m_genres_encoded"].values)
])
y_pred = model.predict(X_infer)

1/1 [==============================] - 0s 34ms/step


In [ ]:
# print min_max scalaer and encoder 
print("Min-Max Scaler for Ratings:", rating_min_max_scaler.data_min_, rating_min_max_scaler.data_max_)
print("Min-Max Scaler for Age and Occupation:", user_min_max_scaler.data_min_, user_min_max_scaler.data_max_)
print("Label Encoder for Gender:", gender_encoder.classes_)
print("One-Hot Encoded Genres:", list(unique_genres))
print("Shape of model input:", X.shape)

Min-Max Scaler for Ratings: [1.] [5.]
Min-Max Scaler for Age and Occupation: [1. 0.] [56. 20.]
Label Encoder for Gender: ['F' 'M']
One-Hot Encoded Genres: ['Thriller', 'Crime', 'Documentary', 'Western', 'Film-Noir', 'Comedy', 'Animation', 'Fantasy', 'Action', 'Romance', 'Musical', 'Horror', 'Mystery', 'Sci-Fi', 'Adventure', 'War', "Children's", 'Drama']
Shape of model input: (1000209, 21)


In [ ]:
# template 5
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy

# load data
df_user = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_user""")
df_movie = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_movie""")
df_rating = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_rating""")

rating_min_max_scaler = MinMaxScaler()
df_rating['r_rating_encoded'] = rating_min_max_scaler.fit_transform(df_rating[['r_rating']])

# construct a user-movie ratings matrix
# user_id: [1, 6040]
# movie_id: [1, 3706]
# user_movie_ratings = np.zeros((df_user.shape[0], df_movie.shape[0]))
# for _, row in tqdm(df_rating.iterrows(), total=df_rating.shape[0]):
#     user_index = int(row['r_user_id'] - 1)  # assuming user_id starts from 1
#     movie_index = int(row['r_movie_id'] - 1)  # assuming movie_id starts from 1
#     user_movie_ratings[user_index, movie_index] = row['r_rating_encoded']

user_movie_ratings = df_rating.pivot(index='r_user_id', columns='r_movie_id', values='r_rating_encoded').fillna(0).values

user_min_max_scaler = MinMaxScaler()
user_gender_encoder = LabelEncoder()
df_user[['u_age_encoded', 'u_occupation_encoded']] = user_min_max_scaler.fit_transform(df_user[['u_age', 'u_occupation']])
df_user['u_gender_encoded'] = user_gender_encoder.fit_transform(df_user['u_gender'])

X = df_user[["u_age_encoded", "u_occupation_encoded", "u_gender_encoded"]].values
y = user_movie_ratings
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X.shape[1],)),
        tf.keras.layers.Dense(1024, activation="relu"),
        tf.keras.layers.Dense(2048, activation="relu"),
        tf.keras.layers.Dense(y.shape[1], activation="sigmoid"),  # Output layer for all movies
    ]
)
model.compile(
    optimizer="adam",
    loss=BinaryCrossentropy(from_logits=False),
    metrics=['mse', 'mae'],
)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train, epochs=10, batch_size=256, validation_data=(X_val, y_val))

# testing part 
df_user_infer = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_user""")

df_user_infer[['u_age_encoded', 'u_occupation_encoded']] = user_min_max_scaler.transform(df_user_infer[['u_age', 'u_occupation']])
df_user_infer["u_gender_encoded"] = user_gender_encoder.transform(df_user_infer["u_gender"])
X_infer = df_user_infer[["u_age_encoded", "u_occupation_encoded", "u_gender_encoded"]].values
y_pred = model.predict(X_infer)

In [111]:
print("Min-Max Scaler for Age and Occupation:", user_min_max_scaler.data_min_, user_min_max_scaler.data_max_)
print("Label Encoder for gender:", user_gender_encoder.classes_)

Min-Max Scaler for Age and Occupation: [1. 0.] [56. 20.]
Label Encoder for gender: ['F' 'M']


In [ ]:
# template 6
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder
from surprise import SVD
from surprise import Dataset
from surprise.reader import Reader

# load data
df_user = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_user""")
df_movie = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_movie""")
df_rating = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_rating""")

# info of SVD model: https://surprise.readthedocs.io/en/stable/matrix_factorization.html
reader = Reader()
data_train = Dataset.load_from_df(
    df_rating[['r_user_id', 'r_movie_id', 'r_rating']], reader)
svd = SVD()
trainset = data_train.build_full_trainset()
model = svd.fit(trainset)

In [ ]:
# SVD has the following parameters:
print("bu: ", model.bu.shape)
print("bi: ", model.bi.shape)
print("pu: ", model.pu.shape)
print("qi: ", model.qi.shape)

In [17]:
# testing part
df_test = utils.fetch_data_from_postgres_via_psycopg2(
  """
  select u_user_id, m_movie_id
  from movielens_user
  cross join
  movielens_movie
  limit 10;
""")

preds = []
for _, row in tqdm(df_test.iterrows(), total=df_test.shape[0]):
    user_id = row['u_user_id']
    movie_id = row['m_movie_id']
    pred_rating = model.predict(user_id, movie_id).est
    preds.append(pred_rating)

  0%|          | 0/10 [00:00<?, ?it/s]